# Lesson 0 - Part 2: Why RAG Works

By now, you understand:
- ✅ How to call Claude via API
- ✅ What tokens are
- ✅ How embeddings work

**Now the big question: what's the problem we're solving?**

Claude is incredibly smart, but it has a major limitation: its knowledge was frozen during training.

If you ask it about your company's internal handbook, it has no idea.
If you ask it about something that happened last week, it doesn't know.

**Enter: RAG (Retrieval-Augmented Generation)**

The idea is simple:
1. Find relevant information (**retrieval**)
2. Give it to Claude (**augmentation**)
3. Let Claude generate the answer (**generation**)

This lesson explores why RAG is so powerful and when it works best.

## The Problem: Hallucination Without Context

Let's see what happens when we ask Claude something it doesn't know:

In [2]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Ask Claude about something very specific (a made-up company policy)
question = "What is the parental leave policy at TechCorp Industries?"

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": question}]
)

print("❌ Without context:")
print(response.content[0].text)
print()
print("Notice: Claude either makes up an answer or says it doesn't know.")
print("This is the hallucination problem.")

❌ Without context:
I don't have access to current information about TechCorp Industries' specific parental leave policy. To find this information, I'd recommend:

1. **Company website** - Check their careers or HR/benefits page
2. **Glassdoor or Indeed** - Employee reviews often mention leave policies
3. **Contact HR directly** - Call or email their human resources department
4. **LinkedIn** - Some companies post benefits information on their company page
5. **Job postings** - Often include benefits details

If you're considering working there or need this info for another reason, contacting their HR department would give you the most accurate and up-to-date details.

Is there anything else I can help you with?

Notice: Claude either makes up an answer or says it doesn't know.
This is the hallucination problem.


## The Solution: Give Claude Context

Now let's give Claude the actual information first:

In [3]:
# First, we have the information (imagine we retrieved this from a document)
handbook_excerpt = """
TECHCORP INDUSTRIES - EMPLOYEE HANDBOOK

Section 5: Time Off and Benefits

Parental Leave Policy:
- Employees are entitled to 16 weeks of paid parental leave
- This applies to birth parents, adoptive parents, and non-birthing partners
- Leave can be taken within 12 months of birth/adoption
- 100% salary is maintained during leave
- Benefits continue during leave period
- Additional unpaid leave up to 6 months available upon request
"""

# Now ask Claude the same question WITH context
prompt_with_context = f"""Context from the TechCorp handbook:
{handbook_excerpt}

Question: {question}

Answer based ONLY on the context provided."""

response_with_context = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt_with_context}]
)

print("✅ With context (from handbook):")
print(response_with_context.content[0].text)
print()
print("Now Claude gives an accurate answer based on the provided context!")

✅ With context (from handbook):
# TechCorp Industries Parental Leave Policy

Based on the handbook, TechCorp Industries offers the following parental leave benefits:

- **16 weeks of paid parental leave**
- **Eligibility:** Birth parents, adoptive parents, and non-birthing partners
- **Timing:** Leave can be taken within 12 months of birth or adoption
- **Salary:** 100% salary is maintained during the leave period
- **Benefits:** Health and other benefits continue during leave
- **Additional option:** Up to 6 months of unpaid leave is available upon request

Now Claude gives an accurate answer based on the provided context!


## The RAG Loop

This is the core idea:

```
User asks: "What's our parental leave policy?"
     ↓
1. RETRIEVE: Find relevant documents (semantic search)
     ↓
Found: "Section 5: Time Off and Benefits"
     ↓
2. AUGMENT: Add to prompt
     ↓
Send to Claude: "Context: [documents] Question: [user query]"
     ↓
3. GENERATE: Claude generates answer
     ↓
Claude: "16 weeks paid leave..."
```

**Key insight:** RAG lets Claude act as a reasoning engine over your specific information.

It doesn't need to memorize everything—it just needs to find the right info and reason about it.

## RAG vs. Other Approaches

You might be thinking: "Why not just fine-tune Claude on our documents?"

Good question! Let's compare:

| Approach | Speed | Cost | Update | Quality | Use Case |
|----------|-------|------|--------|---------|----------|
| **RAG** | ⚡ Fast | 💰 Cheap | 🔄 Instant | 📊 High | Most use cases |
| **Fine-tuning** | 🐢 Slow | 💸 Expensive | ⏳ Weeks | 🎯 Domain-specific | Only if RAG insufficient |
| **Prompt only** | ⚡ Fast | 💰 Cheap | 🔄 Instant | ❌ Poor | Simple Q&A |

**Why RAG wins:**
- 🎯 **Accuracy**: Claude reasons over your actual documents
- 💰 **Cost**: No expensive training
- 📚 **Flexibility**: Change documents anytime
- 🚀 **Speed**: No training time
- 🔐 **Privacy**: Documents stay on your system (if self-hosted)

## When RAG Breaks (Preview)

RAG is powerful, but it's not magic. Here are common failure modes we'll solve in later lessons:

### 1. **Wrong Information Retrieved**
Problem: You ask "What's our parental leave?", but retrieve docs about vacation policies.

Why: Naive keyword matching ("leave" appears in both)

Solution: Better retrieval (semantic search + better ranking)

### 2. **Context Confusion**
Problem: Claude says "According to the handbook..." but doesn't tell you which section.

Why: You gave it documents but no attribution

Solution: Add metadata (source, section, date)

### 3. **Hallucination Under Pressure**
Problem: Claude retrieves docs but still makes up details.

Why: If context is ambiguous, Claude fills gaps with its training knowledge

Solution: Better prompts + agentic reasoning ("I don't know" is okay)

### 4. **Retrieval Misses**
Problem: The information exists but wasn't retrieved.

Why: Poor chunking or embedding quality

Solution: Better chunking strategy + hybrid search

**Don't worry—we'll solve all of these in the next lessons!**

## Key Takeaway

RAG is a game-changer because:

1. **Context is king**: Claude is great at reasoning; just give it the right info
2. **It's fast**: No training needed
3. **It's cheap**: Compared to alternatives
4. **It's flexible**: Update documents anytime

In the next notebook, we'll learn **how to actually retrieve those documents** using ChromaDB and embeddings.

Ready to build? Let's go! 🚀